# Choquet integral classifier workflow

This classifier applies a threshold directly to `C_nu(x)`. A non-additive capacity gives a piecewise-linear decision surface, and the capacity plus threshold are fitted by genetic search under 0-1 loss.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

from capacities_ml_fin.base.capacities import VariableUniverse
from capacities_ml_fin.base.interpretation import pairwise_interactions, shapley_indices
from capacities_ml_fin.ml.model_selection import capacity_parameter_grid
from capacities_ml_fin.ml.models import ChoquetClassifier
from capacities_ml_fin.ml.optimization import L1Penalty, L2Penalty, Solver
from capacities_ml_fin.ml.preprocessing import CapacityNormalizer

rng = np.random.default_rng(23)


## 1. Binary data

The target is a threshold event with a small amount of label noise. Inputs remain in their original units until the preprocessing stage.


In [2]:
grid = np.linspace(0.05, 0.95, 12)
profitability_score, liquidity_score = np.meshgrid(grid, grid)
profitability_score = profitability_score.ravel()
liquidity_score = liquidity_score.ravel()
n = profitability_score.size
X = pd.DataFrame(
    {
        "profitability": 5.0 + 20.0 * profitability_score,
        "liquidity": 0.8 + 1.7 * liquidity_score,
        "volatility": rng.uniform(0.10, 0.60, n),
    }
)
oriented = np.column_stack(
    (
        (X["profitability"] - 5.0) / 20.0,
        (X["liquidity"] - 0.8) / 1.7,
        (0.60 - X["volatility"]) / 0.50,
    )
)
utility = np.minimum(oriented[:, 0], oriented[:, 1])
y = (utility >= 0.35).astype(int)
flip = rng.choice(n, size=4, replace=False)
y[flip] = 1 - y[flip]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7
)
display(X.head())
print("Class counts:", np.bincount(y))


,profitability,liquidity,volatility
0,6.000000,0.885,0.446967
1,7.636364,0.885,0.420729
2,9.272727,0.885,0.164322
3,10.909091,0.885,0.156854
4,12.545455,0.885,0.426673


Class counts: [80 64]


## 2. Normalize inside a pipeline and select interaction order

Cross-validation compares an additive capacity with a 2-additive capacity. Accuracy matches the classifier's hard 0-1 objective.


In [3]:
universe = VariableUniverse(tuple(X.columns))
solver_options = {
    "population_size": 50,
    "n_generations": 60,
    "seed": 23,
    "verbose": False,
    "equality_tolerance": 1e-4,
}
pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            ChoquetClassifier(
                universe=universe,
                solver=Solver.PYMOO,
                solver_options=solver_options,
            ),
        ),
    ]
)
search = GridSearchCV(
    pipeline,
    capacity_parameter_grid(parameter_name="model__sparsity", orders=(1, 2)),
    scoring="balanced_accuracy",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=4),
    n_jobs=1,
)
search.fit(X_train, y_train)

selection_results = pd.DataFrame(search.cv_results_)
selection_results["mean_validation_balanced_accuracy"] = selection_results["mean_test_score"]
display(
    selection_results[
        [
            "param_model__sparsity",
            "mean_validation_balanced_accuracy",
            "std_test_score",
        ]
    ]
)
print("Selected capacity:", search.best_params_["model__sparsity"])


,param_model__sparsity,mean_validation_balanced_accuracy,std_test_score
0,"KAdditivity(order=1, shape=<CapacityShape.GENE...",0.800000,0.040505
1,"KAdditivity(order=2, shape=<CapacityShape.GENE...",0.758333,0.051116


Selected capacity: KAdditivity(order=1, shape=<CapacityShape.GENERAL: 'general'>)


## 3. Regularize and refit

Structural regularization through k-additivity is the main complexity control. If interactions are selected, L1 shrinks their Möbius terms; if cross-validation selects an additive capacity, L2 stabilizes its singleton weights.


In [4]:
best_sparsity = search.best_params_["model__sparsity"]
compilation = best_sparsity.compile(universe.n_elements)
interaction_positions = np.array(
    [i for i, mask in enumerate(compilation.bundle.parameter_masks) if mask.bit_count() >= 2],
    dtype=int,
)
if interaction_positions.size:
    penalty = L1Penalty(weight=0.02, selection=interaction_positions)
    print("Using L1 regularization on interaction coefficients")
else:
    singleton_positions = np.arange(compilation.bundle.n_parameters, dtype=int)
    penalty = L2Penalty(weight=0.02, selection=singleton_positions)
    print("Using L2 regularization on additive singleton weights")

final_pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            ChoquetClassifier(
                universe=universe,
                sparsity=best_sparsity,
                penalty=penalty,
                solver=Solver.PYMOO,
                solver_options=solver_options,
            ),
        ),
    ]
).fit(X_train, y_train)


Using L2 regularization on additive singleton weights


## 4. Out-of-sample evaluation

All reported scores are better when higher. Balanced accuracy is useful when class sizes differ. `predict_proba` contains hard class indicators here, not calibrated probabilities.


In [5]:
fitted_model = final_pipeline.named_steps["model"]
prediction = final_pipeline.predict(X_test)
scores = fitted_model.decision_function(
    final_pipeline.named_steps["normalize"].transform(X_test)
)
majority_class = int(np.bincount(y_train).argmax())
baseline_prediction = np.full(y_test.shape, majority_class, dtype=int)

def classification_metrics(observed, predicted):
    return {
        "accuracy": accuracy_score(observed, predicted),
        "balanced_accuracy": balanced_accuracy_score(observed, predicted),
        "precision": precision_score(observed, predicted, zero_division=0),
        "recall": recall_score(observed, predicted, zero_division=0),
        "F1": f1_score(observed, predicted, zero_division=0),
    }

evaluation = pd.DataFrame(
    {
        "Choquet classifier": classification_metrics(y_test, prediction),
        "majority baseline": classification_metrics(y_test, baseline_prediction),
    }
)
display(evaluation)
print(f"Learned threshold: {fitted_model.threshold_:.6f}")
display(pd.DataFrame(confusion_matrix(y_test, prediction), index=["true 0", "true 1"], columns=["pred 0", "pred 1"]))
display(pd.DataFrame({"score": scores[:8], "observed": y_test[:8], "predicted": prediction[:8]}))


,Choquet classifier,majority baseline
accuracy,0.805556,0.555556
balanced_accuracy,0.800000,0.500000
precision,0.800000,0.000000
recall,0.750000,0.000000
F1,0.774194,0.000000


Learned threshold: 0.547823


,pred 0,pred 1
true 0,17,3
true 1,4,12


,score,observed,predicted
0,0.396118,1,0
1,0.493950,0,0
2,0.268463,0,0
3,0.289849,0,0
4,0.730822,1,1
5,0.441780,0,0
6,0.372029,0,0
7,0.594087,1,1


## 5. Interpret the fitted capacity


In [6]:
display(pd.Series(shapley_indices(fitted_model.capacity_), name="Shapley importance"))
display(pd.Series(pairwise_interactions(fitted_model.capacity_), name="pairwise interaction"))


profitability    0.378570
liquidity        0.374187
volatility       0.247290
Name: Shapley importance, dtype: float64

profitability  liquidity     0.0
               volatility    0.0
liquidity      volatility    0.0
Name: pairwise interaction, dtype: float64

## 6. Classify new observations


In [7]:
X_new = pd.DataFrame(
    {
        "profitability": [8.0, 17.0, 24.0],
        "liquidity": [1.0, 1.7, 2.3],
        "volatility": [0.55, 0.32, 0.14],
    }
)
new_normalized = final_pipeline.named_steps["normalize"].transform(X_new)
display(
    X_new.assign(
        score=fitted_model.decision_function(new_normalized),
        predicted_class=final_pipeline.predict(X_new),
    )
)


,profitability,liquidity,volatility,score,predicted_class
0,8.0,1.0,0.55,0.094173,0
1,17.0,1.7,0.32,0.570439,1
2,24.0,2.3,0.14,0.955013,1
